In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langsmith import traceable
from langchain_mcp_adapters.client import MultiServerMCPClient
from dotenv import load_dotenv
import asyncio
import os


In [ ]:
load_dotenv()

api_key = os.getenv('GOOGLE_API_KEY')
model_name = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')

llm = ChatGoogleGenerativeAI(
    model=model_name,
    api_key=api_key,
)

client = MultiServerMCPClient(
    {
        'arith': {
            'transport': 'stdio',
            'command': os.getenv('MCP_PYTHON', 'python'),
            'args': [os.getenv('MCP_SERVER_PATH', os.path.join('..', 'Python-101', 'mcp', 'calculator_server.py'))],
        }
    }
)


In [ ]:
# tools
search_tool = DuckDuckGoSearchRun(region='us-en')

@tool
@traceable
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """Perform a basic arithmetic operation on two numbers."""
    try:
        if operation == 'add':
            result = first_num + second_num
        elif operation == 'subtract':
            result = first_num - second_num
        elif operation == 'multiply':
            result = first_num * second_num
        elif operation == 'divide':
            result = first_num / second_num
        else:
            return {'error': f'Unsupported operation: {operation}. '
                             'Supported operations are add, subtract, multiply, divide.'}
        return {
            'first_num': first_num,
            'second_num': second_num,
            'operation': operation,
            'result': result,
        }
    except ZeroDivisionError:
        return {'error': 'Division by zero is not allowed.'}
    except Exception as e:
        return {'error': str(e)}


In [ ]:
tools = [search_tool, calculator]

llm_with_tools = llm.bind_tools(tools)


In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [ ]:
async def build_graph():
    # mcp tools must be fetched asynchronously before the graph is built
    mcp_tools = await client.get_tools()
    all_tools = [search_tool, calculator, *mcp_tools]
    llm_with_all_tools = llm.bind_tools(all_tools)

    async def chat_node(state: ChatState):
        messages = state['messages']
        response = await llm_with_all_tools.ainvoke(messages)
        return {'messages': [response]}

    tool_node = ToolNode(all_tools)

    graph = StateGraph(ChatState)
    graph.add_node('chat_node', chat_node)
    graph.add_node('tools', tool_node)
    graph.add_edge(START, 'chat_node')
    graph.add_conditional_edges('chat_node', tools_condition)
    graph.add_edge('tools', 'chat_node')
    return graph.compile()


async def main():
    chatbot = await build_graph()

    result = await chatbot.ainvoke({'messages': [HumanMessage(
        content='What is the current cricket headline today? Calculate the sum of 40 and 30.'
    )]})
    print(result['messages'][-1].content)

if __name__ == '__main__':
    asyncio.run(main())
